<a href="https://colab.research.google.com/github/api-sage/boston-housing-price-predictor/blob/feature%2Fdl-model/model/boston_housing_predictor_dl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modules importation

In [10]:
import numpy as np
from numpy import ndarray
from typing import List

In [11]:
def assert_same_shape(array: ndarray, array_grad: ndarray) -> None:
  assert array.shape == array_grad.shape, \
  '''
  Two ndarrays should have the same shape;
  Instead, the first ndarray's shape is {0}
  while the second's shape is {1}
  '''.format(tuple(array.shape), tuple(array_grad.shape))
  return None

In [12]:
class Operation(object):

  def __init__(self):
    pass

  def forward(self, input_: ndarray) -> ndarray:
    # Stores input_ in the input_ variable and calls the _output function
    self.input_ = input_
    self.output = self._output()
    return self.output

  def backward(self, output_grad: ndarray) -> ndarray:
    '''
    Calls the self._input_grad method and check appropriate shape.
    '''
    assert_same_shape(self.output, output_grad)
    self.input_grad = self._input_grad(output_grad)
    assert_same_shape(self.input_, self.input_grad)
    return self.input_grad

  def _output(self) -> ndarray:
    # This method must be defined for each operation
    raise NotImplementedError()

  def _input_grad(self, output_grad: ndarray) -> ndarray:
    # This method must be defined for each operation
    raise NotImplementedError()

In [13]:
class ParamOperation(Operation):
  '''
  An operation with parameters
  '''

  def __init__(self, param: ndarray):
    super().__init__()
    self.param = param

  def backward(self, output_grad: ndarray) -> ndarray:

    assert_same_shape(self.output, output_grad)
    self.input_grad = self._input_grad(output_grad)
    self.param_grad = self._param_grad(output_grad)

    return self.input_grad

  def _param_grad(self, output_grad: ndarray) -> ndarray:
    '''
    Every subclass of ParamOperation must implement _param_grad
    '''
    return NotImplementedError()

In [14]:
class WeightMultiply(ParamOperation):
  '''
  Weight multiplication operation for a neural network
  '''
  def __inint__(self, W: ndarray):
    super().__init__(W)

  def _output(self) -> ndarray:
    '''
    Computes output
    '''
    return np.dot(self.input_, self.param)

  def _input_grad(self, output_grad: ndarray) -> ndarray:
    '''
    Computes input gradient
    '''
    return np.dot(output_grad, np.transpose(self.param, (1,0)))

  def _param_grad(self, output_grad: ndarray) -> ndarray:
    '''
    Computes parameter gradient
    '''
    return np.dot(np.transpose(self.input_, (1,0)), output_grad)

In [15]:
class BiasOperation(ParamOperation):
  '''
  Compute bias addition
  '''
  def __init__(self, B: ndarray):
    '''
    Initialize Operation with self.param as B
    Check appropriate shape
    '''
    assert B.shape[0] == B.shape[1] == 1, \
    '''
    Bias array must be of shape 1.
    What was passed in has a shape of {0}'''.format(B.shape)
    super().__init__(B)

  def _output(self) -> ndarray:
    '''
    Compute output
    '''
    return self.input_ + self.param

  def _input_grad(self, output_grad: ndarray) -> ndarray:
    '''
    Compute input gradient
    '''
    return np.ones_like(self.input_) * output_grad

  def _param_grad(self, output_grad: ndarray) -> ndarray:
    '''
    Compute param gradient
    '''
    param_grad = np.ones_like(self.param) * output_grad
    return np.sum(param_grad, axis=0).reshape(1, param_grad.shape[1])

In [16]:
class Sigmoid(Operation):
  '''
  Sigmoid activation function
  '''
  def __init__(self):
    super().__init__

  def _output(self) -> ndarray:
    '''
    Computes output
    '''
    return 1.0/(1.0 + np.exp(-1.0 * self.input_))

  def __input_grad(self, output_grad: ndarray) -> ndarray:
    '''
    Computes sigmoid input gradient
    '''
    sigmoid_backward = self.output * (1.0 - self.output)
    input_grad = sigmoid_backward * output_grad
    return input_grad